# Chương 2: Thực hành - Tác tử và Giải quyết bài toán bằng Tìm kiếm

**Mục tiêu bài thực hành:**
1. Xây dựng môi trường mê cung lưới 2D (`GridWorld Maze`) với các loại địa hình có chi phí di chuyển khác nhau (đường bằng phẳng vs đầm lầy).
2. Tự cài đặt từ đầu (**from scratch**) các thuật toán tìm kiếm mù: **BFS** (Breadth-First Search) và **UCS** (Uniform-Cost Search / Dijkstra).
3. Cài đặt thuật toán tìm kiếm có thông tin **A\*** với hàm Heuristic Manhattan và Heuristic Euclidean; đối chiếu số lượng nút duyệt và chi phí đường đi với BFS/UCS.
4. Cài đặt giải thuật tìm kiếm cục bộ **Luyện kim mô phỏng (Simulated Annealing)** giải bài toán Người du lịch ghé thăm các trạm kiểm soát trong mê cung.
5. Cài đặt mô phỏng cơ chế **Tìm kiếm chùm (Beam Search)** trong giải mã chuỗi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import heapq

# Cố định seed ngẫu nhiên
np.random.seed(42)

## 1. Xây dựng Môi trường Mê cung (GridWorld Environment)

- Bản đồ kích thước $20 \times 20$.
- Ô trống thông thường (giá trị `0`): Chi phí bước đi $c = 1$.
- Ô đầm lầy (giá trị `2`): Chi phí bước đi $c = 5$ (địa hình hiểm trở, tiêu hao năng lượng).
- Ô tường cản (giá trị `1`): Không thể đi qua.

In [ ]:
class GridMaze:
    def __init__(self, width=20, height=20, wall_ratio=0.22, swamp_ratio=0.15):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)
        
        # Tạo tường chướng ngại vật ngẫu nhiên
        walls = np.random.rand(height, width) < wall_ratio
        self.grid[walls] = 1
        
        # Tạo các vùng đầm lầy (swamp)
        swamps = (np.random.rand(height, width) < swamp_ratio) & (~walls)
        self.grid[swamps] = 2
        
        # Đảm bảo điểm bắt đầu (0, 0) và điểm đích (height-1, width-1) luôn thông suốt
        self.start = (0, 0)
        self.goal = (height - 1, width - 1)
        self.grid[self.start] = 0
        self.grid[self.goal] = 0
        
    def is_valid(self, r, c):
        return 0 <= r < self.height and 0 <= c < self.width and self.grid[r, c] != 1
        
    def get_neighbors(self, state):
        r, c = state
        neighbors = []
        # 4 hướng di chuyển: Lên, Xuống, Trái, Phải
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if self.is_valid(nr, nc):
                neighbors.append((nr, nc))
        return neighbors
        
    def get_cost(self, next_state):
        # Chi phí bước đi vào ô tiếp theo
        r, c = next_state
        if self.grid[r, c] == 2:
            return 5.0 # Ô đầm lầy
        return 1.0     # Ô thông thường

maze = GridMaze(width=20, height=20, wall_ratio=0.20, swamp_ratio=0.15)
print(f"Tạo mê cung kích thước {maze.width}x{maze.height}")
print(f"Xuất phát: {maze.start}, Đích: {maze.goal}")

## 2. Cài đặt các thuật toán Tìm kiếm mù (BFS & UCS)

- **BFS:** Sử dụng hàng đợi FIFO, mở rộng theo tầng, chỉ tối ưu về *số bước đi*.
- **UCS:** Sử dụng hàng đợi ưu tiên theo chi phí tích lũy $g(n)$, đảm bảo tối ưu về *tổng chi phí năng lượng*.

In [ ]:
def reconstruct_path(came_from, current):
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path

def bfs_search(maze):
    queue = deque([maze.start])
    came_from = {}
    visited = {maze.start}
    explored_nodes = []
    
    while queue:
        current = queue.popleft()
        explored_nodes.append(current)
        
        if current == maze.goal:
            path = reconstruct_path(came_from, current)
            total_cost = sum(maze.get_cost(node) for node in path[1:])
            return path, total_cost, explored_nodes
            
        for neighbor in maze.get_neighbors(current):
            if neighbor not in visited:
                visited.add(neighbor)
                came_from[neighbor] = current
                queue.append(neighbor)
                
    return None, float('inf'), explored_nodes

def ucs_search(maze):
    # Priority queue lưu trữ tuple: (g_cost, counter, current_state)
    pq = []
    counter = 0
    heapq.heappush(pq, (0.0, counter, maze.start))
    came_from = {}
    cost_so_far = {maze.start: 0.0}
    explored_nodes = []
    
    while pq:
        current_cost, _, current = heapq.heappop(pq)
        explored_nodes.append(current)
        
        if current == maze.goal:
            path = reconstruct_path(came_from, current)
            return path, current_cost, explored_nodes
            
        for neighbor in maze.get_neighbors(current):
            new_cost = current_cost + maze.get_cost(neighbor)
            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                came_from[neighbor] = current
                counter += 1
                heapq.heappush(pq, (new_cost, counter, neighbor))
                
    return None, float('inf'), explored_nodes

## 3. Cài đặt thuật toán Tìm kiếm Heuristic A\*

Hàm đánh giá: $f(n) = g(n) + h(n)$
- **Heuristic Manhattan:** $h_1(n) = |r_n - r_G| + |c_n - c_G|$ (chấp nhận được và nhất quán trên lưới 4 hướng).
- **Heuristic Euclidean:** $h_2(n) = \sqrt{(r_n - r_G)^2 + (c_n - c_G)^2}$.

In [ ]:
def heuristic_manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def heuristic_euclidean(a, b):
    return np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

def a_star_search(maze, heuristic_fn=heuristic_manhattan):
    pq = []
    counter = 0
    start = maze.start
    goal = maze.goal
    
    # Lưu (f_cost, counter, current_state)
    h_start = heuristic_fn(start, goal)
    heapq.heappush(pq, (h_start, counter, start))
    came_from = {}
    g_cost = {start: 0.0}
    explored_nodes = []
    visited = set()
    
    while pq:
        _, _, current = heapq.heappop(pq)
        
        if current in visited:
            continue
        visited.add(current)
        explored_nodes.append(current)
        
        if current == goal:
            path = reconstruct_path(came_from, current)
            return path, g_cost[current], explored_nodes
            
        for neighbor in maze.get_neighbors(current):
            tentative_g = g_cost[current] + maze.get_cost(neighbor)
            if neighbor not in g_cost or tentative_g < g_cost[neighbor]:
                g_cost[neighbor] = tentative_g
                f_score = tentative_g + heuristic_fn(neighbor, goal)
                came_from[neighbor] = current
                counter += 1
                heapq.heappush(pq, (f_score, counter, neighbor))
                
    return None, float('inf'), explored_nodes

## 4. Thực thi và Đối chiếu hiệu năng giữa các giải thuật

In [ ]:
bfs_path, bfs_cost, bfs_explored = bfs_search(maze)
ucs_path, ucs_cost, ucs_explored = ucs_search(maze)
astar_path, astar_cost, astar_explored = a_star_search(maze, heuristic_manhattan)

print("=== BẢNG SO SÁNH HIỆU NĂNG TÌM KIẾM MÊ CUNG ===")
print(f"BFS  : Chi phí = {bfs_cost:.1f}, Số bước = {len(bfs_path) if bfs_path else 0}, Số nút đã duyệt = {len(bfs_explored)}")
print(f"UCS  : Chi phí = {ucs_cost:.1f}, Số bước = {len(ucs_path) if ucs_path else 0}, Số nút đã duyệt = {len(ucs_explored)}")
print(f"A*   : Chi phí = {astar_cost:.1f}, Số bước = {len(astar_path) if astar_path else 0}, Số nút đã duyệt = {len(astar_explored)}")

# Nhận xét: UCS và A* cùng tìm ra chi phí tối ưu, nhưng A* mở rộng số nút ít hơn đáng kể nhờ Heuristic định hướng!

## 5. Trực quan hóa đường đi và Vùng không gian đã khám phá

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
algs = [
    ("BFS (Chiều rộng)", bfs_path, bfs_explored, bfs_cost),
    ("UCS (Chi phí đồng nhất)", ucs_path, ucs_explored, ucs_cost),
    ("A* (Heuristic Manhattan)", astar_path, astar_explored, astar_cost)
]

for idx, (title, path, explored, cost) in enumerate(algs):
    ax = axes[idx]
    # Vẽ nền bản đồ: Trắng (Đường trống), Đen (Tường), Vàng nhạt (Đầm lầy)
    bg = np.ones((maze.height, maze.width, 3))
    bg[maze.grid == 1] = [0.15, 0.15, 0.15]   # Tường
    bg[maze.grid == 2] = [0.85, 0.75, 0.45]   # Đầm lầy
    
    ax.imshow(bg, origin='upper')
    
    # Vẽ các nút đã duyệt (màu xanh dương nhạt)
    if explored:
        exp_r, exp_c = zip(*explored)
        ax.scatter(exp_c, exp_r, c='cyan', s=15, alpha=0.35, label=f'Đã duyệt: {len(explored)}')
        
    # Vẽ đường đi tìm được (màu đỏ rực)
    if path:
        pr, pc = zip(*path)
        ax.plot(pc, pr, color='crimson', linewidth=3, label=f'Đường đi (Chi phí: {cost:.1f})')
        
    # Đánh dấu Start (Xanh lá) và Goal (Đỏ)
    ax.scatter(maze.start[1], maze.start[0], c='lime', s=120, edgecolors='k', zorder=5, label='Start')
    ax.scatter(maze.goal[1], maze.goal[0], c='red', s=120, edgecolors='k', zorder=5, label='Goal')
    
    ax.set_title(f"{title}\n{len(explored)} nút duyệt | Chi phí: {cost:.1f}", fontsize=12, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc='upper left', fontsize=8)

plt.suptitle("So sánh Không gian Tìm kiếm và Quỹ đạo: BFS vs UCS vs A*", fontsize=15, y=0.98)
plt.tight_layout()
plt.show()

## 6. Tìm kiếm cục bộ: Luyện kim mô phỏng (Simulated Annealing) giải bài toán TSP

Giả sử tác tử cần ghé thăm 10 trạm kiểm soát rải rác trên bản đồ rồi quay về điểm xuất phát.  
Không gian hoán vị có $10! / 2 \approx 1.8 \times 10^6$ hành trình. Luyện kim mô phỏng tìm hành trình ngắn nhất bằng cách:
- Bắt đầu từ một tour ngẫu nhiên với năng lượng $E = \text{tổng khoảng cách}$.
- Tại mỗi bước, tạo một tour láng giềng bằng phép đảo đoạn (2-opt swap).
- Nếu $\Delta E < 0$: Luôn chấp nhận.
- Nếu $\Delta E \ge 0$: Chấp nhận với xác suất $P = \exp(-\Delta E / T)$.

In [ ]:
# Tạo tọa độ của 10 trạm kiểm soát ngẫu nhiên
n_cities = 10
cities = np.random.uniform(2, 18, size=(n_cities, 2))

def calculate_total_distance(tour, cities):
    dist = 0.0
    for i in range(len(tour)):
        c1 = cities[tour[i]]
        c2 = cities[tour[(i + 1) % len(tour)]]
        dist += np.linalg.norm(c1 - c2)
    return dist

def simulated_annealing_tsp(cities, initial_temp=100.0, cooling_rate=0.995, max_iters=2500):
    current_tour = list(range(len(cities)))
    np.random.shuffle(current_tour)
    current_dist = calculate_total_distance(current_tour, cities)
    
    best_tour = list(current_tour)
    best_dist = current_dist
    
    T = initial_temp
    history_dist = [current_dist]
    
    for it in range(max_iters):
        # 2-opt swap: Chọn ngẫu nhiên 2 vị trí và đảo ngược đoạn con
        i, j = np.random.choice(len(cities), size=2, replace=False)
        if i > j:
            i, j = j, i
        new_tour = current_tour[:i] + current_tour[i:j+1][::-1] + current_tour[j+1:]
        new_dist = calculate_total_distance(new_tour, cities)
        
        delta_E = new_dist - current_dist
        
        # Tiêu chuẩn Metropolis-Hastings
        if delta_E < 0 or np.random.rand() < np.exp(-delta_E / max(T, 1e-6)):
            current_tour = new_tour
            current_dist = new_dist
            if current_dist < best_dist:
                best_dist = current_dist
                best_tour = list(current_tour)
                
        T *= cooling_rate
        history_dist.append(current_dist)
        
    return best_tour, best_dist, history_dist

best_tour, best_dist, dist_history = simulated_annealing_tsp(cities)
print(f"Khoảng cách ban đầu: {dist_history[0]:.2f} -> Khoảng cách tối ưu sau SA: {best_dist:.2f}")

# Trực quan hóa kết quả luyện kim
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Đồ thị hội tụ năng lượng
ax1.plot(dist_history, color='purple', linewidth=1.5)
ax1.set_title("Quá trình Hội tụ Năng lượng (Tổng khoảng cách)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Số vòng lặp")
ax1.set_ylabel("Tổng chiều dài tour")

# Đồ thị hành trình tốt nhất
tour_coords = cities[best_tour + [best_tour[0]]]
ax2.plot(tour_coords[:, 0], tour_coords[:, 1], 'o-r', markersize=8, linewidth=2, label='Đường đi tối ưu')
for idx, (x, y) in enumerate(cities):
    ax2.text(x+0.3, y+0.3, f"T{idx}", fontsize=11, fontweight='bold')
ax2.set_title(f"Hành trình Tối ưu TSP (Khoảng cách = {best_dist:.2f})", fontsize=12, fontweight='bold')
ax2.set_xlim(0, 20)
ax2.set_ylim(0, 20)
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Tìm kiếm chùm (Beam Search) trong AI hiện đại

Mô phỏng bài toán sinh chuỗi token: Tại mỗi bước thời gian, mô hình ngôn ngữ xuất ra phân phối xác suất trên tập từ vựng.  
- **Greedy Search (K=1):** Luôn chọn token có xác suất cao nhất tại bước hiện tại.
- **Beam Search (K=3):** Duy trì 3 giả thuyết có tích xác suất (tổng log-probability) cao nhất.

In [ ]:
# Giả lập ma trận xác suất logarit của 5 bước sinh chuỗi, từ vựng gồm 4 từ: ['A', 'B', 'C', 'D']
vocab = ['A', 'B', 'C', 'D']
log_probs_matrix = [
    [-0.5, -1.2, -2.5, -3.0], # Bước 1: Từ 'A' chiếm ưu thế
    [-2.0, -0.2, -1.5, -3.5], # Bước 2: Từ 'B' chiếm ưu thế
    [-1.0, -1.8, -0.4, -2.2], # Bước 3: Từ 'C' chiếm ưu thế
    [-0.3, -2.1, -1.9, -1.5], # Bước 4: Từ 'A' chiếm ưu thế
]

def greedy_search_demo(log_probs_matrix, vocab):
    sequence = []
    total_log_prob = 0.0
    for step_probs in log_probs_matrix:
        best_idx = np.argmax(step_probs)
        sequence.append(vocab[best_idx])
        total_log_prob += step_probs[best_idx]
    return " ".join(sequence), total_log_prob

def beam_search_demo(log_probs_matrix, vocab, beam_width=3):
    # Mỗi phần tử trong chùm là tuple: (tổng_log_prob, danh_sách_tokens)
    beams = [(0.0, [])]
    
    for step_probs in log_probs_matrix:
        all_candidates = []
        for score, seq in beams:
            for word_idx, log_p in enumerate(step_probs):
                candidate = (score + log_p, seq + [vocab[word_idx]])
                all_candidates.append(candidate)
                
        # Giữ lại top K chùm tốt nhất
        ordered = sorted(all_candidates, key=lambda x: x[0], reverse=True)
        beams = ordered[:beam_width]
        
    best_score, best_seq = beams[0]
    return " ".join(best_seq), best_score, beams

greedy_seq, greedy_score = greedy_search_demo(log_probs_matrix, vocab)
beam_seq, beam_score, all_top_beams = beam_search_demo(log_probs_matrix, vocab, beam_width=3)

print("=== SO SÁNH GIẢI MÃ CHUỖI: GREEDY VS BEAM SEARCH ===")
print(f"Greedy Search : Chuỗi = '{greedy_seq}', Tổng Log-Prob = {greedy_score:.3f}")
print(f"Beam Search(3): Chuỗi = '{beam_seq}', Tổng Log-Prob = {beam_score:.3f}")
print("Top 3 chùm ứng viên của Beam Search:")
for rank, (score, seq) in enumerate(all_top_beams, 1):
    print(f"  Hạng {rank}: {' '.join(seq)} (Điểm: {score:.3f})")

## 8. Đúc kết bài học thực hành

1. **BFS:** Đảm bảo tìm được đường đi có ít bước nhất, nhưng không xét đến chi phí địa hình và làm bùng nổ số nút lưu trong bộ nhớ RAM.
2. **UCS (Dijkstra):** Đảm bảo nghiệm tối ưu về chi phí tích lũy $g(n)$, nhưng phải duyệt không gian theo hình tròn đồng mức mở rộng, tốn nhiều phép duyệt.
3. **A\* Search:** Nhờ có hàm Heuristic $h(n)$ định hướng về phía mục tiêu, A* tạo ra "hình phễu" hướng thẳng tới đích, cắt giảm đáng kể không gian nút cần duyệt trong khi vẫn bảo toàn nghiệm tối ưu tuyệt đối.
4. **Luyện kim mô phỏng (Simulated Annealing):** Minh chứng khả năng giải quyết các bài toán tối ưu hóa tổ hợp phi tuyến tính (TSP) mà không bị kẹt ở cực tiểu địa phương.
5. **Beam Search:** Cầu nối giải thuật tìm kiếm với kỷ nguyên xử lý ngôn ngữ tự nhiên và AI tạo sinh hiện đại.